# 03 — Dynamics Exploration

Compare mechanism combinations: homophily, triadic closure, attention budgets.

In [ ]:
import numpy as np
from abm_core import init_hyperbolic_uniform, burt_constraint
from abm_dynamics import (
    mechanism_homophily, mechanism_triadic_closure,
    mechanism_attention_budget,
)
from abm_runner import run_simulation
import matplotlib.pyplot as plt
import networkx as nx

N = 300
TARGET_K = 10
ALPHA = 0.5

rng = np.random.default_rng(42)
init = init_hyperbolic_uniform(N, TARGET_K, rng, alpha=ALPHA)
print(f"Initialized {N} agents, R={init.metadata['R']:.2f}")

In [ ]:
configs = {
    "Homophily only": [
        lambda s: mechanism_homophily(s, lam=3.0),
    ],
    "Homophily + Triadic": [
        lambda s: mechanism_homophily(s, lam=3.0),
        lambda s: mechanism_triadic_closure(s, tau=1.5),
    ],
    "Homophily + Budget": [
        lambda s: mechanism_homophily(s, lam=3.0),
        lambda s: mechanism_attention_budget(s, beta=3.0),
    ],
    "All three": [
        lambda s: mechanism_homophily(s, lam=3.0),
        lambda s: mechanism_triadic_closure(s, tau=1.5),
        lambda s: mechanism_attention_budget(s, beta=3.0),
    ],
}

results = {}
for name, mechanisms in configs.items():
    history = run_simulation(
        init_result=init,
        mechanisms=mechanisms,
        budgets=np.full(N, 10),
        n_steps=200,
        rng=np.random.default_rng(42),
    )
    results[name] = history
    print(f"{name}: final edges={history.stats[-1]['n_edges']}, "
          f"mean_degree={history.stats[-1]['mean_degree']:.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for name, history in results.items():
    ts = [s["t"] for s in history.stats]
    axes[0].plot(ts, [s["mean_degree"] for s in history.stats], label=name)
    axes[1].plot(ts, [s["n_edges"] for s in history.stats], label=name)

# Compute clustering over time for each config
for name, history in results.items():
    clustering = []
    for A in history.frames[::10]:  # sample every 10 steps
        G = nx.from_numpy_array(A)
        clustering.append(nx.average_clustering(G))
    axes[2].plot(range(0, len(history.frames), 10), clustering, label=name)

axes[0].set_xlabel("Timestep")
axes[0].set_ylabel("Mean degree")
axes[0].legend(fontsize=8)
axes[0].set_title("Mean Degree")

axes[1].set_xlabel("Timestep")
axes[1].set_ylabel("Number of edges")
axes[1].legend(fontsize=8)
axes[1].set_title("Edge Count")

axes[2].set_xlabel("Timestep")
axes[2].set_ylabel("Avg clustering coefficient")
axes[2].legend(fontsize=8)
axes[2].set_title("Clustering")

plt.tight_layout()
plt.show()

In [ ]:
# Network snapshots for "All three"
history = results["All three"]
snapshot_times = [0, 50, 100, 200]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
coords = init.viz_coords
pos = {i: coords[i] for i in range(N)}

for ax, t in zip(axes, snapshot_times):
    A = history.frames[t]
    G = nx.from_numpy_array(A)
    degrees = np.array([d for _, d in G.degree()])
    circle = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=1)
    ax.add_patch(circle)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.03, width=0.5)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=10,
                           node_color=degrees, cmap='YlOrRd', alpha=0.7)
    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)
    ax.set_aspect('equal')
    ax.set_title(f't={t}, edges={int(A.sum())//2}')

plt.suptitle("Network Evolution (All mechanisms)")
plt.tight_layout()
plt.show()